<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/13-Adding_Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intelligent Query Routing and Agents

## What This Notebook Covers
This notebook demonstrates how to build intelligent query routing systems using LlamaIndex's `RouterQueryEngine`. You will see how an LLM can automatically decide which knowledge base to consult based on the nature of a question, and how to compose agents that leverage multiple specialized tools.

## Learning Objectives
By the end of this notebook, you will be able to:
- Load a pre-built vector store from HuggingFace Hub and query it with LlamaIndex
- Create a `RouterQueryEngine` that routes queries to the appropriate knowledge base
- Understand how `LLMSingleSelector` makes routing decisions
- Build a `FunctionAgent` that uses multiple query-engine tools
- Configure a multi-LLM agent that routes code questions to GPT-5 and general questions to Gemini

## Prerequisites
- Familiarity with Python and Jupyter notebooks
- Basic understanding of vector stores and retrieval-augmented generation (RAG)
- OpenAI API key and Google API key (set in Colab Secrets)
- Completion of earlier notebooks in this series (RAG with LlamaIndex, Vector Stores)

## Install Packages and Setup Variables


In [ ]:
!pip install -qU llama-index==0.14.16 openai==2.30.0 google-genai==1.70.0 llama-index-embeddings-openai==0.5.2 \
                chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2 opentelemetry-api==1.38.0\
                llama-index-llms-openai==0.6.26 llama-index-llms-google-genai==0.9.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [ ]:
import os

# Set the following API keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_OPENAI_API_KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_GOOGLE_API_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", "")

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

nest_asyncio.apply()

# Initialize Models


In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load Indexes


In [ ]:
# Download vector store from Hugging Face Hub
from huggingface_hub import hf_hub_download

vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

In [ ]:
!unzip vectorstore.zip

Archive:  vectorstore.zip
   creating: ai_tutor_knowledge/
   creating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/length.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/index_metadata.pickle  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/link_lists.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/header.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/data_level0.bin  
  inflating: ai_tutor_knowledge/chroma.sqlite3  


In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Create your index
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
vector_index = VectorStoreIndex.from_vector_store(vector_store)

In [ ]:
# Query engine for the AI tutor knowledge base
ai_tutor_knowledge_query_engine = vector_index.as_query_engine(similarity_top_k=3)

res = ai_tutor_knowledge_query_engine.query("How does Retrieval Augmented Generation (RAG) work?")
print(res.response)

Retrieval-Augmented Generation (RAG) combines an information-retrieval pipeline with a generative large language model to produce more accurate, up-to-date, and grounded responses. It works as a staged system with two main components and several intervening processing steps:

1. Retrieval component
- Indexing: Source documents are organized for efficient search, using sparse (inverted index) or dense (vector embeddings) representations. Documents are usually split into chunks and embedded for semantic comparison.
- Searching: Given a query, the system fetches candidate documents or chunks from the index.
- Reranking (optional): Retrieved candidates are re-ordered by relevance using a reranker to surface the most useful passages.

2. Generation component
- Prompting and inferencing: A large language model (LLM) takes the user query plus the retrieved (and possibly reranked/repacked) content and generates the final response. Prompting techniques (e.g., Chain of Thought, Tree of Thought, 

In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("Metadata\t", src.metadata)
    print("-_" * 20)

Node ID	 2aa05360-f43a-4819-bce7-0acf7b897eab
Title	 Searching for Best Practices in Retrieval-Augmented Generation:1 Introduction
Text	 Generative large language models are prone to producing outdated information or fabricating facts, although they were aligned with human preferences by reinforcement learning [1] or lightweight alternatives [2–5]. Retrieval-augmented generation (RAG) techniques address these issues by combining the strengths of pretraining and retrieval-based models, thereby providing a robust framework for enhancing model performance [6]. Furthermore, RAG enables rapid deployment of applications for specific organizations and domains without necessitating updates to the model parameters, as long as query-related documents are provided. Many RAG approaches have been proposed to enhance large language models (LLMs) through query-dependent retrievals [6–8]. A typical RAG workflow usually contains multiple intervening processing steps: query classification (determining w

# Router

Routers are modules that take in a user query and a set of “choices” (defined by metadata), and returns one or more selected choices.

They can be used for the following use cases and more:

- Selecting the right data source among a diverse range of data sources

- Deciding whether to do summarization (e.g. using summary index query engine) or semantic search (e.g. using vector index query engine)

- Deciding whether to “try” out a bunch of choices at once and combine the results (using multi-routing capabilities).


## Let's create a different query engine with Mistral AI information


In [ ]:
from pathlib import Path
import requests
import time

wiki_titles = [
    "Mistral AI",
    "Llama (language model)",
    "Claude AI",
    "OpenAI",
    "Gemini AI",
]

data_path = Path("llm_data_wiki")
if not data_path.exists():
    data_path.mkdir()

# Set up headers with User-Agent (REQUIRED by Wikipedia API)
headers = {
    'User-Agent': 'YourAppName/1.0 (your-email@example.com)'  # Replace with your info if this dummy gives an error
}

for title in wiki_titles:
    try:
        # Make the request with headers
        response = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "format": "json",
                "titles": title,
                "prop": "extracts",
                "explaintext": True,
            },
            headers=headers  # Add headers here
        )

        # Check if request was successful
        response.raise_for_status()

        if not response.text:
            print(f"Empty response for '{title}'")
            continue

        data = response.json()

        # Extract the page content
        if "query" in data and "pages" in data["query"]:
            page = next(iter(data["query"]["pages"].values()))
            if "extract" in page:
                wiki_text = page["extract"]
                with open(data_path / "llm_data_wiki.txt", "a", encoding="utf-8") as fp:
                    fp.write(f"Title: {title}\n{wiki_text}\n\n")
                print(f"Successfully saved: {title}")
            else:
                print(f"No extract found for '{title}'")
        else:
            print(f"Unexpected response format for '{title}'")
        time.sleep(0.5)

    except requests.exceptions.RequestException as e:
        print(f"Request error for '{title}': {e}")
    except ValueError as e:  # JSON decode error
        print(f"JSON decode error for '{title}': {e}")
        print(f"Response text: {response.text[:200]}...")

Successfully saved: Mistral AI
Successfully saved: Llama (language model)
Successfully saved: Claude AI
Successfully saved: OpenAI
Successfully saved: Gemini AI


In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.text_splitter import TokenTextSplitter
from llama_index.core.extractors import (
    SummaryExtractor,
    QuestionsAnsweredExtractor,
    KeywordExtractor,
)

# Assuming you have prepared a directory for LLM data
documents = SimpleDirectoryReader("llm_data_wiki").load_data()

text_splitter = TokenTextSplitter(separator=" ", chunk_size=512, chunk_overlap=128)

transformations = [
    text_splitter,
    QuestionsAnsweredExtractor(questions=2),
    SummaryExtractor(summaries=["prev", "self"]),
    KeywordExtractor(keywords=10),
    OpenAIEmbedding(model="text-embedding-3-small"),
]

llm_index = VectorStoreIndex.from_documents(documents=documents, transformations=transformations)

llm_query_engine = llm_index.as_query_engine(similarity_top_k=2)

100%|██████████| 44/44 [00:35<00:00,  1.25it/s]


In [ ]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import PydanticSingleSelector
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

# Initialize tools
ai_tutor_knowledge_tool = QueryEngineTool.from_defaults(
    query_engine=ai_tutor_knowledge_query_engine,
    description="Useful for questions about general generative AI concepts",
)
llm_tool = QueryEngineTool.from_defaults(
    query_engine=llm_query_engine,
    description="Useful for questions about particular LLMs like Mistral, Claude, OpenAI, Gemini",
)

# Initialize the router query engine (single selection, LLM-based)
query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        ai_tutor_knowledge_tool,
        llm_tool,
    ],
)

In [ ]:
res = query_engine.query(
    "What is the LLama model?",
)
print(res.response)

Llama (stylized LLaMA) is a family of large language foundation models developed by Meta AI, first announced on February 24, 2023. It was released as a set of models in multiple sizes (designed to be usable on different hardware) and trained only on publicly available data. The project's inference code was published under GPLv3, while access to the original model weights was distributed by application. In evaluations, smaller Llama variants showed strong performance (the 13B model exceeded GPT‑3 on many NLP benchmarks and the 65B model was competitive with state‑of‑the‑art models). Llama has since been the basis for instruction‑tuned variants, community projects, and numerous downstream applications.


In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 6a92397f-3327-4919-ad3f-3229b3fd25b7
Text	 remain online for reference.
Meditron is a family of Llama-based finetuned on a corpus of clinical guidelines, PubMed papers, and articles. It was created by researchers at École Polytechnique Fédérale de Lausanne School of Computer and Communication Sciences, and the Yale School of Medicine. It shows increased performance on medical-related benchmarks such as MedQA and MedMCQA.
Zoom used Meta Llama 2 to create an AI Companion that can summarize meetings, provide helpful presentation tips, and assist with message responses. This AI Companion is powered by multiple models, including Meta Llama 2.
Reuters reported in 2024 that many Chinese foundation models relied on Llama models for their training.


=== llama.cpp ===

Software developer Georgi Gerganov released llama.cpp as open-source on March 10, 2023. It's a re-implementation of Llama in C++, allowing systems without a powerful GPU to run the model locally. The llama.cpp project in

In [ ]:
res = query_engine.query("Explain parameter-efficient finetuning methods")
print(res.response)

Parameter-efficient fine-tuning (PEFT) refers to methods that adapt large pretrained models to new tasks while modifying only a small fraction of parameters (or adding a small set) instead of updating every weight. The main approaches and representative techniques are:

1. Selective fine-tuning
- What it is: Only a subset of the model’s existing parameters are updated (e.g., a subset of layers or specific parameter groups).
- When to use: When you want very simple changes and prefer to keep most of the pretrained knowledge fixed.
- Trade-offs: Lower memory and compute than full fine-tuning, but effectiveness depends on which parameters are selected.

2. Reparameterization (low-rank adaptation)
- What it is: Replace or augment weight updates with a low-rank factorization so the model learns small matrices that approximate the full weight change. LoRA (Low-Rank Adaptation) is a common example: the weight update ΔW is represented as B·A where A and B are low-rank.
- How it reduces paramet

In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 6be88fa3-2f8b-43e7-aba0-d874b39809fc
Text	 # FourierFT: Discrete Fourier Transformation Fine-Tuning[FourierFT](https://huggingface.co/papers/2405.03003) is a parameter-efficient fine-tuning technique that leverages Discrete Fourier Transform to compress the model's tunable weights. This method outperforms LoRA in the GLUE benchmark and common ViT classification tasks using much less parameters.FourierFT currently has the following constraints:- Only `nn.Linear` layers are supported.- Quantized layers are not supported.If these constraints don't work for your use case, consider other methods instead.The abstract from the paper is:> Low-rank adaptation (LoRA) has recently gained much interest in fine-tuning foundation models. It effectively reduces the number of trainable parameters by incorporating low-rank matrices A and B to represent the weight change, i.e., Delta W=BA. Despite LoRA's progress, it faces storage challenges when handling extensive customization adaptations or 

## Optional: How the Router Makes Decisions

In [ ]:
# The LLMSingleSelector uses an LLM to decide which tool to route to.
# We can inspect the selector's reasoning by examining the source nodes.
test_cases = [
    ("What is the LLaMA model?",                  "→ Expected: LLaMA knowledge base"),
    ("What is Mistral AI's architecture?",         "→ Expected: Mistral Wikipedia"),
    ("What is the capital of France?",             "→ Expected: Neither (off-topic)"),
]

print("=== Router Routing Tests ===\n")
for question, expected in test_cases:
    try:
        r = query_engine.query(question)
        print(f"Q: {question}")
        print(f"   {expected}")
        print(f"   Response: {str(r)[:200]}...")
        print()
    except Exception as e:
        print(f"Q: {question} → Error: {e}\n")

=== Router Routing Tests ===

Q: What is the LLaMA model?
   → Expected: LLaMA knowledge base
   Response: LLaMA (often written Llama or LLaMA) is a family of large language models developed by Meta AI. Key points:

- First announced February 24, 2023, as a set of foundation models trained only on publicly...

Q: What is Mistral AI's architecture?
   → Expected: Mistral Wikipedia
   Response: The available document does not state a specific model architecture (e.g., decoder-only transformer, encoder–decoder, mixture-of-experts, etc.). It does describe Mistral AI as a developer of large lan...

Q: What is the capital of France?
   → Expected: Neither (off-topic)
   Response: I don't have the information needed to answer that question. If you allow the use of external/general knowledge, I can provide the answer....



# Function Agent using OpenAI GPT 5 Model


In [ ]:
system_message_openai_agent = """You are a highly knowledgeable assistant specialized in Artificial Intelligence. Your primary role is to assist users by providing accurate, detailed, and context-specific responses using the tools available.

Your answers are aimed to teach students, so they should be complete, clear, and easy to understand.

Use the available tools to gather insights pertinent to the field of AI. Always use two tools at the same time. These tools accept a string (a user query rewritten as a statement) and return informative content regarding the domain of AI.
e.g:
User question: 'How can I fine-tune an LLM?'
Input to the tool: 'Fine-tuning an LLM'

User question: How can quantize an LLM?
Input to the tool: 'Quantization for LLMs'

User question: 'Teach me how to build an AI agent"'
Input to the tool: 'Building an AI Agent'

Only some information returned by the tools might be relevant to the question, so ignore the irrelevant part and answer the question with what you have.

Your responses are exclusively based on the output provided by the tools. Refrain from incorporating information not directly obtained from the tool's responses.

When the conversation deepens or shifts focus within a topic, adapt your input to the tools to reflect these nuances. This means if a user requests further elaboration on a specific aspect of a previously discussed topic, you should reformulate your input to the tool to capture this new angle or more profound layer of inquiry.

Provide comprehensive answers, ideally structured in multiple paragraphs, drawing from the tool's variety of relevant details. The depth and breadth of your responses should align with the scope and specificity of the information retrieved.

Should the tools repository lack information on the queried topic, politely inform the user that the question transcends the bounds of your current knowledge base, citing the absence of relevant content in the tool's documentation.

At the end of your answers, always invite the students to ask deeper questions about the topic if they have any. Make sure to reformulate the question to the tool to capture this new angle or more profound layer of inquiry.

Do not refer to the documentation directly, but use the information provided within it to answer questions.

If code is provided in the information, share it with the students. It's important to provide complete code blocks so they can execute the code when they copy and paste them.

Make sure to format your answers in Markdown format, including code blocks and snippets.

Politely reject questions not related to AI, while being cautious not to reject unfamiliar terms or acronyms too quickly."""

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

# Initialize the LLM
llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})

# Create the FunctionAgent
agent = FunctionAgent(
    tools=[ai_tutor_knowledge_tool, llm_tool],
    llm=llm,
    system_prompt=system_message_openai_agent,
    verbose=False
)

# Run the agent queries
import asyncio

async def run_agent(query):
    response = await agent.run(query)
    return response

In [ ]:
# Execute the async function
response = asyncio.run(run_agent("What is the LLama model?"))

print(response)

Here’s a clear, student-friendly summary of the LLaMA (LLaMA / Llama) family based on the retrieved material.

What LLaMA is (core idea)
- LLaMA models are a family of autoregressive, decoder‑only Transformer language models developed by Meta.  
- They follow typical Transformer design but use specific implementation choices: SwiGLU activations, rotary positional embeddings (RoPE), and RMSNorm instead of classic LayerNorm.  
- Meta emphasized scaling training-token volume (and careful dataset curation) as a primary lever for improving performance.

Major releases and highlights
- LLaMA (LLaMA 1)
  - Announced Feb 2023. Multiple sizes (notably 13B and 65B).
  - Trained on large public corpora (~1.4 trillion tokens in reported reproductions).
  - The 13B model outperformed GPT‑3 on many benchmarks; 65B was competitive with contemporaries.

- LLaMA 2
  - Released July 2023 (7B, 13B, 70B variants).
  - Trained on ~2 trillion tokens with dataset curation (reduced exposure risk, upsampled tr

In [ ]:
# Execute the async function
response = asyncio.run(run_agent("Explain parameter-efficient finetuning methods"))

print(response)

I’m sorry — my current tool sources don’t contain information about parameter‑efficient fine‑tuning (LoRA, adapters, prefix‑tuning, prompt‑tuning, low‑rank adaptation, etc.). The parallel tool calls returned either no relevant content or only unrelated fine‑tuning/RLHF notes (supervised fine‑tuning details, RLHF workflow and tricks), so I can’t produce an accurate, tool‑based explanation of PEFT methods from the available repository.

I can do one of the following — please pick which you prefer:
- I can search external references and summarize parameter‑efficient fine‑tuning methods (LoRA, adapters, prefix/prompt tuning, BitFit, etc.) with examples and code.
- I can provide a high‑level explanation based only on general knowledge (not drawn from the tool outputs).
- I can instead explain the fine‑tuning / RLHF details my tools do contain (supervised fine‑tuning setup, rejection sampling before PPO, separate reward models for safety/helpfulness, and “Ghost attention” for multi‑turn syst

In [ ]:
# Execute the async function
response = asyncio.run(run_agent("Write the recipe for a chocolate cake."))

print(response)

I don’t have a chocolate cake recipe available in the materials I just checked. I can generate one for you, but first I need a few quick preferences so I make the recipe the way you want.

Please tell me:
- Any dietary restrictions? (none, vegan, gluten-free, dairy-free, egg-free, etc.)
- Desired size / format: 8" round (2 layers), 9x13 sheet, or cupcakes (12)?
- Do you want frosting? If so: chocolate buttercream, ganache, cream cheese, or none?
- Any extra features: molten center, coffee-flavored, dense fudgy vs. light and fluffy?

Examples of how I’ll reformulate this to my sources/tools once you answer:
- "Chocolate cake recipe — 8\" round, 2-layer, chocolate buttercream, no dietary restrictions"
- "Chocolate cake recipe — vegan cupcakes with chocolate ganache"
- "Chocolate cake recipe — 9x13, gluten-free, dense fudgy with coffee"

Tell me your choices and I’ll write the full recipe (ingredients, step-by-step instructions, baking times, and tips) in a copy-paste-ready format. Ask an

# Code-Related Questions to GPT-5, the Remaining Questions to Gemini

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import PydanticSingleSelector
from llama_index.core.tools import QueryEngineTool

# Initialize LLMs
gpt_5_llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})

gemini_llm = GoogleGenAI(model="gemini-2.5-flash", temperature=1, max_tokens=512)

# Define query engines
llm_query_engine_code = vector_index.as_query_engine(
    llm=gpt_5_llm,
    similarity_top_k=3,
    embed_model=OpenAIEmbedding(model="text-embedding-3-small", mode="text_search"),
)

llm_query_engine_rest = vector_index.as_query_engine(
    llm=gemini_llm,
    similarity_top_k=3,
    embed_model=OpenAIEmbedding(model="text-embedding-3-small", mode="text_search"),
)

# Define tools for the LLM
llm_tool_code = QueryEngineTool.from_defaults(
    query_engine=llm_query_engine_code,
    description="Ideal for handling code-related queries, technical implementations, and troubleshooting involving Large Language Models.",
    name="LLMCodeTool",
)

llm_tool_rest = QueryEngineTool.from_defaults(
    query_engine=llm_query_engine_rest,
    description="Best suited for answering conceptual, theoretical, and general questions about Large Language Models.",
    name="LLMGeneralTool",

)


system_message_openai_agent_tools = """
You are a highly knowledgeable assistant specialized in Large Language Models. Your primary role is to assist users by providing accurate, detailed, and context-specific responses. You have access to two specialized tools:

1. **LLMCodeTool** – Use this tool when the query involves code-related tasks, technical implementations, debugging, or troubleshooting issues in code.
2. **LLMGeneralTool** – Use this tool for answering conceptual, theoretical, or general questions about Large Language Models that do not involve code specifics.

When a query is received:
- First, decide which tool best fits the user's request.
- If the question is technical or code-oriented, route the query to LLMCodeTool.
- If the question is more general or conceptual, route the query to LLMGeneralTool.
- If the query does not clearly fall into either category, provide a direct answer using your own capabilities.

Always ensure your responses are clear, concise, and directly address the user's needs. Maintain a professional tone and provide detailed explanations where necessary.
"""
# Create the FunctionAgent
agent = FunctionAgent(
    tools=[llm_tool_code, llm_tool_rest],
    llm=gpt_5_llm,
    system_prompt=system_message_openai_agent_tools,
    verbose=False
)

# Run the agent queries
import asyncio

async def run_agent(query):
    response = await agent.run(query)

    return response.tool_calls[0].tool_name



In [ ]:
# Execute the async function
response_code = asyncio.run(run_agent("How do I fine-tune the LLama model? Write the code for it"))

print(response_code)

LLMCodeTool


In [ ]:
response_general = asyncio.run(run_agent("What is the relationship between Llama models and Meta"))

print(response_general)

LLMGeneralTool


## Optional: Tracing Agent Tool Calls

In [ ]:
# Run a few queries through the agent and print which tool was invoked.
# The FunctionAgent from LlamaIndex logs tool calls in the response metadata.
import asyncio

test_queries = [
    "What is the context length of the LLaMA 2 70B model?",
    "What is 2 + 2?",  # Off-topic — agent should say it can't answer
]

async def run_traced_agent(q):
    handler = agent.run(q)
    response = await handler
    print(f"Q: {q}")
    print(f"A: {str(response)[:300]}")
    print()

print("=== Agent Query Traces ===\n")
for q in test_queries:
    try:
        asyncio.run(run_traced_agent(q))
    except Exception as e:
        print(f"Q: {q} → Error: {e}\n")

=== Agent Query Traces ===

Q: What is the context length of the LLaMA 2 70B model?
A: The official LLaMA 2 70B model uses a 4,096-token context window. 

Note: community/third‑party builds or fine‑tuned variants may provide extended windows (e.g., 32k) via modified position encodings or inference-time techniques, but the released base LLaMA 2 70B checkpoints are 4,096 tokens.

Q: What is 2 + 2?
A: 2 + 2 = 4.

